# MSCS 634 — Advanced Data Mining for Data-Driven Insights and Predictive Modeling
## Project Deliverable 1: Data Collection, Cleaning, and Exploration

**Name:** Shilpa Mesineni  
**Course:** MSCS 634 – Advanced Data Mining and Big Data Analytics  
**Deliverable:** Project 1 — Data Collection, Cleaning, and Exploration

---
## 1. Dataset Selection and Justification

### Dataset: Pima Indians Diabetes Dataset
**Source:** UCI Machine Learning Repository / Kaggle  
**URL:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

### Why this dataset?
The Pima Indians Diabetes dataset is an excellent choice for this project for the following reasons:

- **Size:** 768 records — well above the 500+ record requirement
- **Attributes:** 9 attributes (8 features + 1 target) — meets the 8-10 attribute requirement
- **Domain relevance:** Healthcare data directly relevant to chronic disease prediction, a critical real-world problem
- **Richness:** Contains physiological measurements (glucose, BMI, insulin, blood pressure) that allow meaningful EDA and modeling
- **Known challenges:** Contains zero-coded missing values (e.g., Glucose=0, BMI=0) that require careful preprocessing — ideal for demonstrating data cleaning skills
- **Versatility:** Supports regression (predicting glucose), classification (predicting diabetes outcome), and clustering in future deliverables

### Feature Descriptions
| Feature | Description | Unit |
|---------|-------------|------|
| Pregnancies | Number of times pregnant | Count |
| Glucose | Plasma glucose concentration (2-hour oral glucose tolerance test) | mg/dL |
| BloodPressure | Diastolic blood pressure | mm Hg |
| SkinThickness | Triceps skinfold thickness | mm |
| Insulin | 2-hour serum insulin | μU/mL |
| BMI | Body mass index (weight/height²) | kg/m² |
| DiabetesPedigreeFunction | Diabetes hereditary likelihood score | Score |
| Age | Age of the patient | Years |
| Outcome | Diabetes diagnosis (1=positive, 0=negative) | Binary |

In [ ]:
# ── Import all required libraries ────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

print('All libraries imported successfully.')

---
## 2. Load Dataset and Inspect Structure

In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────────
# Download from Kaggle or UCI; file should be in the same directory as this notebook.
# URL: https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, header=None, names=columns)

print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# ── Display first 5 rows ──────────────────────────────────────────────────────
print('First 5 rows of the raw dataset:')
df.head()

In [ ]:
# ── Dataset info: data types and non-null counts ──────────────────────────────
print('=== Dataset Info ===')
df.info()
print()
print('=== Statistical Summary (Raw Data) ===')
df.describe().round(2)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
outcome_counts = df['Outcome'].value_counts()
print('Class Distribution:')
print(f'  Non-Diabetic (0): {outcome_counts[0]} ({outcome_counts[0]/len(df)*100:.1f}%)')
print(f'  Diabetic     (1): {outcome_counts[1]} ({outcome_counts[1]/len(df)*100:.1f}%)')
print(f'\nClass imbalance ratio: {outcome_counts[0]/outcome_counts[1]:.2f}:1')

---
## 3. Data Cleaning

### 3.1 Identify True Missing Values (Zero-Coded)

A key issue with this dataset: several features use **0 as a placeholder for missing values**, which is biologically impossible:
- `Glucose = 0` → impossible (a person cannot have zero plasma glucose)
- `BloodPressure = 0` → impossible
- `SkinThickness = 0` → biologically implausible
- `Insulin = 0` → missing (very common in this dataset)
- `BMI = 0` → impossible

We replace these zeros with `NaN` for honest missing value treatment.

In [ ]:
# ── Step 1: Replace biologically impossible zeros with NaN ────────────────────
zero_not_valid = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

df_clean = df.copy()
df_clean[zero_not_valid] = df_clean[zero_not_valid].replace(0, np.nan)

# Count and display missing values per column BEFORE imputation
missing_before = df_clean.isnull().sum()
missing_pct    = (missing_before / len(df_clean) * 100).round(1)

missing_df = pd.DataFrame({
    'Missing Count': missing_before,
    'Missing %':     missing_pct
})
print('=== Missing Values After Zero-Replacement ===')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# ── Visualize missing value pattern ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
missing_pct_nonzero = missing_pct[missing_pct > 0]
bars = ax.barh(missing_pct_nonzero.index, missing_pct_nonzero.values,
               color=['#E07B54' if v > 30 else '#5B9BD5' for v in missing_pct_nonzero.values],
               edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, missing_pct_nonzero.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Missing Values (%)')
ax.set_title('Missing Value Percentage by Feature (after zero-replacement)', fontsize=12)
ax.set_xlim(0, 55)
plt.tight_layout()
plt.savefig('missing_values.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: Insulin has the highest missingness (48.7%) — median imputation by Outcome class is preferred.')

In [ ]:
# ── Step 2: Impute missing values with class-wise median ─────────────────────
# Reason: Using the median by Outcome class (diabetic vs non-diabetic) preserves
# the biological difference between groups rather than flattening it with a
# single global median.

for col in zero_not_valid:
    df_clean[col] = df_clean.groupby('Outcome')[col].transform(
        lambda x: x.fillna(x.median())
    )

print('=== Missing Values After Imputation ===')
print(df_clean.isnull().sum())
print('\nAll missing values have been imputed successfully.')

In [ ]:
# ── Step 3: Check and remove duplicates ──────────────────────────────────────
n_duplicates = df_clean.duplicated().sum()
print(f'Duplicate rows found: {n_duplicates}')

if n_duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f'Removed {n_duplicates} duplicate rows.')
else:
    print('No duplicate rows — no action required.')

print(f'\nDataset shape after cleaning: {df_clean.shape}')

In [ ]:
# ── Step 4: Detect and cap outliers using IQR ─────────────────────────────────
# Strategy: Winsorize (cap) rather than remove — preserving record count.
# Values below Q1 - 3*IQR or above Q3 + 3*IQR are extreme outliers.

features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

outlier_report = []
df_capped = df_clean.copy()

for col in features:
    Q1  = df_capped[col].quantile(0.25)
    Q3  = df_capped[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    n_out = ((df_capped[col] < lower) | (df_capped[col] > upper)).sum()
    if n_out > 0:
        df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
    outlier_report.append({'Feature': col, 'Q1': round(Q1,2), 'Q3': round(Q3,2),
                           'IQR': round(IQR,2), 'Lower Fence': round(lower,2),
                           'Upper Fence': round(upper,2), 'Outliers Capped': n_out})

out_df = pd.DataFrame(outlier_report)
print('=== IQR Outlier Analysis (3×IQR rule) ===')
print(out_df.to_string(index=False))

In [ ]:
# ── Statistical summary after all cleaning steps ─────────────────────────────
print('=== Statistical Summary After Cleaning ===')
df_capped.describe().round(2)

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1 Distribution of all features ─────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
axes = axes.flatten()
all_cols = features + ['Outcome']
colors = sns.color_palette('muted', len(all_cols))

for i, col in enumerate(all_cols):
    if col == 'Outcome':
        axes[i].bar(['Non-Diabetic (0)', 'Diabetic (1)'],
                    df_capped[col].value_counts().sort_index(),
                    color=['#5B9BD5', '#E07B54'], edgecolor='black', linewidth=0.5)
        axes[i].set_title('Outcome (Target)', fontsize=10)
    else:
        axes[i].hist(df_capped[col], bins=25, color=colors[i],
                     edgecolor='black', linewidth=0.4, alpha=0.85)
        axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions — Pima Indians Diabetes Dataset', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: Glucose and BMI appear roughly bell-shaped; Insulin and Pregnancies are right-skewed.')

In [ ]:
# ── 4.2 Boxplots by Outcome class ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.flatten()
palette = {0: '#5B9BD5', 1: '#E07B54'}

for i, col in enumerate(features):
    sns.boxplot(data=df_capped, x='Outcome', y=col, palette=palette, ax=axes[i],
                linewidth=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticklabels(['Non-Diabetic', 'Diabetic'], fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Feature Distribution by Diabetes Outcome (0=No, 1=Yes)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots_by_outcome.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: Diabetic patients show markedly higher Glucose, BMI, Age, and Insulin levels.')

In [ ]:
# ── 4.3 Correlation heatmap ───────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
corr = df_capped.corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: Glucose has the highest correlation with Outcome (0.49).')
print('Age and Pregnancies are moderately correlated (0.54), which is biologically expected.')
print('SkinThickness and BMI are positively correlated (0.57) — both measure body fat.')

In [ ]:
# ── 4.4 Pairplot for top correlated features ──────────────────────────────────
top_features = ['Glucose', 'BMI', 'Age', 'Insulin', 'Outcome']
pair_df = df_capped[top_features].copy()
pair_df['Outcome'] = pair_df['Outcome'].map({0: 'Non-Diabetic', 1: 'Diabetic'})

g = sns.pairplot(pair_df, hue='Outcome', palette={'Non-Diabetic': '#5B9BD5', 'Diabetic': '#E07B54'},
                 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20})
g.fig.suptitle('Pairplot: Top 4 Features by Diabetes Outcome', fontsize=12, y=1.02)
plt.savefig('eda_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Insight: Diabetic patients cluster at higher Glucose and BMI values across all pairings.')

In [ ]:
# ── 4.5 Scatter plots — key relationships ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette_scatter = {0: '#5B9BD5', 1: '#E07B54'}

pairs = [('Glucose', 'BMI'), ('Age', 'Glucose'), ('BMI', 'Insulin')]
for ax, (xcol, ycol) in zip(axes, pairs):
    for outcome, color in palette_scatter.items():
        subset = df_capped[df_capped['Outcome'] == outcome]
        label = 'Non-Diabetic' if outcome == 0 else 'Diabetic'
        ax.scatter(subset[xcol], subset[ycol], c=color, label=label, alpha=0.45, s=20)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(f'{xcol} vs {ycol}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Scatter Plots: Key Feature Relationships by Outcome', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_scatter_plots.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: High Glucose + High BMI is a strong indicator of diabetes (top-right clusters in red).')

In [ ]:
# ── 4.6 Age group analysis ────────────────────────────────────────────────────
df_capped['AgeGroup'] = pd.cut(df_capped['Age'],
                                bins=[20, 30, 40, 50, 60, 90],
                                labels=['21-30', '31-40', '41-50', '51-60', '60+'])

age_diabetes = df_capped.groupby('AgeGroup', observed=True)['Outcome'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Diabetes rate by age group
axes[0].bar(age_diabetes['AgeGroup'].astype(str), age_diabetes['Outcome'] * 100,
            color='#E07B54', edgecolor='black', linewidth=0.5, alpha=0.85)
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Diabetes Rate (%)')
axes[0].set_title('Diabetes Rate by Age Group', fontsize=11)
for i, val in enumerate(age_diabetes['Outcome']):
    axes[0].text(i, val*100 + 0.5, f'{val*100:.1f}%', ha='center', fontsize=9)

# Glucose mean by age group and outcome
gluc_age = df_capped.groupby(['AgeGroup', 'Outcome'], observed=True)['Glucose'].mean().reset_index()
for out, color, label in [(0, '#5B9BD5', 'Non-Diabetic'), (1, '#E07B54', 'Diabetic')]:
    subset = gluc_age[gluc_age['Outcome'] == out]
    axes[1].plot(subset['AgeGroup'].astype(str), subset['Glucose'],
                 marker='o', color=color, label=label, linewidth=2)
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Mean Glucose (mg/dL)')
axes[1].set_title('Mean Glucose by Age Group and Outcome', fontsize=11)
axes[1].legend()

plt.suptitle('Age Group Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_age_analysis.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: Diabetes rate increases sharply with age — from 24% in 21-30 to 58% in 51-60.')
print('Diabetic patients consistently show higher mean glucose across ALL age groups.')

In [ ]:
# ── 4.7 Outlier visualization — boxplots before and after capping ─────────────
fig, axes = plt.subplots(2, 4, figsize=(15, 8))

for i, col in enumerate(features):
    row = i // 4
    col_idx = i % 4
    data_pairs = [df_clean[col], df_capped[col]]
    bp = axes[row, col_idx].boxplot(data_pairs, labels=['Before\nCapping', 'After\nCapping'],
                                     patch_artist=True,
                                     boxprops=dict(facecolor='lightsteelblue'),
                                     medianprops=dict(color='red', linewidth=2))
    axes[row, col_idx].set_title(col, fontsize=10)

plt.suptitle('Outlier Capping: Before vs After (3×IQR Rule)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_outlier_comparison.png', dpi=110, bbox_inches='tight')
plt.show()
print('Insight: Insulin and DiabetesPedigreeFunction had the most extreme outliers.')
print('After capping, distributions are more stable for downstream modeling.')

---
## 5. Key Insights and Implications for Future Modeling

In [ ]:
# ── Summary of cleaned dataset ────────────────────────────────────────────────
print('=== CLEANED DATASET SUMMARY ===')
print(f'Final shape         : {df_capped.shape[0]} rows × {df_capped.shape[1]} columns')
print(f'Missing values      : {df_capped.isnull().sum().sum()}')
print(f'Duplicate rows      : {df_capped.duplicated().sum()}')
print(f'Diabetic patients   : {(df_capped["Outcome"]==1).sum()} ({(df_capped["Outcome"]==1).mean()*100:.1f}%)')
print(f'Non-diabetic        : {(df_capped["Outcome"]==0).sum()} ({(df_capped["Outcome"]==0).mean()*100:.1f}%)')
print()
print('=== TOP CORRELATIONS WITH OUTCOME ===')
corr_with_outcome = df_capped.corr()['Outcome'].drop('Outcome').abs().sort_values(ascending=False)
print(corr_with_outcome.round(3))
print()
print('=== KEY EDA INSIGHTS ===')
insights = [
    '1. Glucose is the strongest predictor of diabetes (corr=0.49) — will be primary feature in regression.',
    '2. BMI and Age are the next most important features for predicting diabetes outcome.',
    '3. Insulin had 48.7% missing values — class-wise median imputation preserves group differences.',
    '4. Dataset has a 35:65 class imbalance — SMOTE or class weighting may be needed for classification.',
    '5. SkinThickness and BMI are correlated (0.57) — multicollinearity to monitor in regression models.',
    '6. Age and Pregnancies are moderately correlated (0.54) — one may be redundant in full models.',
    '7. Diabetes rate rises sharply with age (24% for 21-30 vs 58% for 51-60).',
]
for ins in insights:
    print(f'  {ins}')

In [ ]:
# ── Save cleaned dataset for use in Project 2 ─────────────────────────────────
df_capped.to_csv('pima_diabetes_cleaned.csv', index=False)
print('Cleaned dataset saved as: pima_diabetes_cleaned.csv')
print('This file will be used as the starting point for Project Deliverable 2.')